<a href="https://colab.research.google.com/github/fanTaux/GoogleCollabQC2025/blob/main/5_Pencari_Email_untuk_Sertifikat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import io

print("Silakan upload 2 file Anda (CSV atau XLSX).")
print("1. File Database Lengkap (berisi kolom 'NIM', 'Name', 'Prodi', 'Cluster', 'Group', 'Email').")
print("2. File Pencarian (berisi kolom 'NIM' dari maba yang dicari).")

try:
    # 1. Proses Upload
    uploaded = files.upload()

    if len(uploaded) != 2:
        print("\n⚠️ Harap upload TEPAT 2 file (CSV atau XLSX) secara bersamaan.")
    else:
        file_names = list(uploaded.keys())
        file1_name = file_names[0]
        file2_name = file_names[1]

        print(f"\nFile '{file1_name}' dan '{file2_name}' berhasil di-upload.")

        def read_file(file_name, uploaded_files):
            file_extension = file_name.split('.')[-1].lower()
            if file_extension == 'csv':
                return pd.read_csv(io.BytesIO(uploaded_files[file_name]))
            elif file_extension in ['xlsx', 'xls']:
                return pd.read_excel(io.BytesIO(uploaded_files[file_name]))
            else:
                raise ValueError(f"Format file tidak didukung: {file_extension}. Harap upload file CSV atau XLSX.")

        try:
            # 2. Membaca File
            df1 = read_file(file1_name, uploaded)
            df2 = read_file(file2_name, uploaded)

            # Standarisasi nama kolom ke huruf KAPITAL semua dan hapus spasi berlebih
            # Agar tidak error jika ada perbedaan 'NIM', 'nim', atau ' Nim '
            df1.columns = df1.columns.str.strip().str.upper()
            df2.columns = df2.columns.str.strip().str.upper()

            # 3. Identifikasi File (Mana DB lengkap, mana DB pencarian)
            if 'EMAIL' in df1.columns:
                df_db = df1
                df_search = df2
                print(f"\n✅ File Database terdeteksi sebagai: {file1_name}")
                print(f"✅ File Pencarian terdeteksi sebagai: {file2_name}")
            elif 'EMAIL' in df2.columns:
                df_db = df2
                df_search = df1
                print(f"\n✅ File Database terdeteksi sebagai: {file2_name}")
                print(f"✅ File Pencarian terdeteksi sebagai: {file1_name}")
            else:
                raise ValueError("Tidak ditemukan kolom 'EMAIL' pada kedua file. Pastikan file database memiliki kolom 'Email'.")

            # Pengecekan kolom NIM
            if 'NIM' not in df_db.columns:
                raise ValueError("File database HARUS memiliki kolom 'NIM'")
            if 'NIM' not in df_search.columns:
                raise ValueError("File pencarian HARUS memiliki kolom 'NIM'")

            # Cek penamaan kolom untuk Nama dan Prodi (menangani bahasa Inggris/Indonesia)
            nama_col = 'NAME' if 'NAME' in df_db.columns else ('NAMA' if 'NAMA' in df_db.columns else None)
            prodi_col = 'PRODI' if 'PRODI' in df_db.columns else ('PROGRAM STUDI' if 'PROGRAM STUDI' in df_db.columns else None)

            # Pastikan NIM bertipe string untuk keperluan pencocokan
            df_search['NIM'] = df_search['NIM'].astype(str).str.strip()
            df_db['NIM'] = df_db['NIM'].astype(str).str.strip()

            # Ambil hanya kolom-kolom yang diperlukan dari DB
            kolom_db_yang_dibawa = ['NIM']
            if nama_col: kolom_db_yang_dibawa.append(nama_col)
            if prodi_col: kolom_db_yang_dibawa.append(prodi_col)
            kolom_db_yang_dibawa.append('EMAIL')

            # Hapus duplikasi di database berdasarkan NIM (untuk amannya)
            df_db_selected = df_db[kolom_db_yang_dibawa].drop_duplicates(subset=['NIM'])

            # Ambil kolom NIM saja dari pencarian dan hapus duplikasi
            df_search_clean = df_search[['NIM']].drop_duplicates(subset=['NIM'])

            # 4. Melakukan Pencocokan Data (Inner/Left Join)
            # Menggunakan left join agar maba yang ada di file kedua tetap muncul (jika tidak ketemu di DB, emailnya akan NaN/kosong)
            hasil_df = pd.merge(df_search_clean, df_db_selected, on='NIM', how='left')

            # 5. Merapikan nama kolom sesuai format akhir
            rename_dict = {'NIM': 'NIM', 'EMAIL': 'Email'}
            if nama_col: rename_dict[nama_col] = 'Nama'
            if prodi_col: rename_dict[prodi_col] = 'Program Studi'

            hasil_df = hasil_df.rename(columns=rename_dict)

            # Menyusun urutan output: Nama - NIM - Program Studi - Email
            kolom_output = []
            if 'Nama' in hasil_df.columns: kolom_output.append('Nama')
            kolom_output.append('NIM')
            if 'Program Studi' in hasil_df.columns: kolom_output.append('Program Studi')
            kolom_output.append('Email')

            hasil_df = hasil_df[kolom_output]

            print("\n✅ Proses pencarian data selesai.")
            print("\nPreview hasil akhir (5 data teratas):")
            print(hasil_df.head())

            # 6. Eksport ke Excel
            output_filename = 'hasil_pencarian_email_maba.xlsx'
            hasil_df.to_excel(output_filename, index=False)

            print(f"\n🎉 File output '{output_filename}' telah dibuat dan siap diunduh.")
            files.download(output_filename)

        except Exception as e:
            print(f"\n❌ Terjadi kesalahan saat membaca atau memproses file: {e}")
            print("Pastikan file memiliki penamaan kolom yang sesuai (NIM, Name/Nama, Prodi/Program Studi, Email).")

except Exception as e:
    print(f"\n❌ Terjadi kesalahan saat proses upload: {e}")

Silakan upload 2 file Anda (CSV atau XLSX).
1. File Database Lengkap (berisi kolom 'NIM', 'Name', 'Prodi', 'Cluster', 'Group', 'Email').
2. File Pencarian (berisi kolom 'NIM' dari maba yang dicari).
